# database check

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "institutional_holding.db"

connection = sqlite3.connect(DB_PATH)
print(f"数据库位置：{DB_PATH}")
print(f"数据库存在：{DB_PATH.exists()}")

数据库位置：d:\KimiData\kimi\workspace\institutional_holding_tracker\data\institutional_holding.db
数据库存在：True


## 查看数据库表

In [2]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)

tables

,name
0,alerts
1,daily_prices
2,fund_holdings
3,holder_mappings
4,holding_changes_summary
5,index_components
6,index_holding_summary
7,indices
8,northbound_holdings
9,sqlite_sequence


## 查看指数基本信息

In [11]:
indices = pd.read_sql_query(
    "SELECT * FROM indices ORDER BY index_code",
    connection,
)

indices

,id,index_name,index_code,exchange,component_count,updated_at


## 查看指数成分股

In [5]:
components = pd.read_sql_query(
    """
    SELECT index_code, stock_code, stock_name, weight, effective_date
    FROM index_components
    ORDER BY index_code, stock_code
    LIMIT 20
    """,
    connection,
)

components

,index_code,stock_code,stock_name,weight,effective_date
0,000300,000001,平安银行,0.433,2026-08-18
1,000300,000002,万科A,0.087,2026-08-18
2,000300,000063,中兴通讯,0.418,2026-08-18
3,000300,000100,TCL科技,0.378,2026-08-18
4,000300,000157,中联重科,0.145,2026-08-18
5,000300,000166,申万宏源,0.162,2026-08-18
6,000300,000301,东方盛虹,0.119,2026-08-18
7,000300,000333,美的集团,1.634,2026-08-18
8,000300,000338,潍柴动力,0.577,2026-08-18
9,000300,000408,藏格矿业,0.243,2026-08-18


In [4]:
component_counts = pd.read_sql_query(
    """
    SELECT index_code, COUNT(DISTINCT stock_code) AS stock_count
    FROM index_components
    GROUP BY index_code
    ORDER BY index_code
    """,
    connection,
)

component_counts

,index_code,stock_count
0,000300,300


## 统计所有指数并随机查看成分股

In [6]:
all_components = pd.read_sql_query(
    """
    SELECT index_code, stock_code, stock_name, weight, effective_date
    FROM index_components
    ORDER BY index_code, stock_code
    """,
    connection,
)

index_summary = (
    all_components.groupby("index_code", as_index=False)
    .agg(stock_count=("stock_code", "nunique"))
    .sort_values("index_code")
)

print(f"实际写入的指数数量：{len(index_summary)}")
index_summary

实际写入的指数数量：1


,index_code,stock_count
0,000300,300


In [7]:
random_samples = pd.concat(
    [
        group.sample(n=min(5, len(group)), random_state=42)
        for _, group in all_components.groupby("index_code")
    ],
    ignore_index=True,
).sort_values(["index_code", "stock_code"])

random_samples

,index_code,stock_code,stock_name,weight,effective_date
3,000300,000408,藏格矿业,0.243,2026-08-18
2,000300,600372,中航机载,0.105,2026-08-18
0,000300,601077,渝农商行,0.138,2026-08-18
4,000300,601633,长城汽车,0.080,2026-08-18
1,000300,603019,中科曙光,0.469,2026-08-18


## 诊断指数基本信息写入

In [8]:
from config.settings import TRACKED_INDICES
from database.db_manager import query_sql
from ingestion.index_components import _update_indices_info

print("配置中的指数：")
for name, info in TRACKED_INDICES.items():
    print(name, info)

_update_indices_info()

updated_indices = pd.DataFrame(query_sql(
    "SELECT * FROM indices ORDER BY index_code"
))
updated_indices

配置中的指数：
沪深300 {'code': '000300', 'exchange': 'sh'}
中证500 {'code': '000905', 'exchange': 'sh'}
创业板指 {'code': '399006', 'exchange': 'sz'}
科创50 {'code': '000688', 'exchange': 'sh'}


,id,index_name,index_code,exchange,component_count,updated_at
0,1,沪深300,000300,sh,300,2026-08-19 09:29:39
1,4,科创50,000688,sh,0,2026-08-19 09:29:39
2,2,中证500,000905,sh,0,2026-08-19 09:29:39
3,3,创业板指,399006,sz,0,2026-08-19 09:29:39


## 回滚本次诊断写入

In [10]:
index_codes = tuple(info["code"] for info in TRACKED_INDICES.values())
placeholders = ", ".join("?" for _ in index_codes)

connection.execute(
    f"DELETE FROM indices WHERE index_code IN ({placeholders})",
    index_codes,
)
connection.commit()

remaining_indices = pd.read_sql_query(
    "SELECT * FROM indices ORDER BY index_code",
    connection,
)
remaining_components = pd.read_sql_query(
    """
    SELECT index_code, COUNT(DISTINCT stock_code) AS stock_count
    FROM index_components
    GROUP BY index_code
    ORDER BY index_code
    """,
    connection,
)

print(f"回滚后 indices 记录数：{len(remaining_indices)}")
print("成分股明细统计：")
remaining_components

回滚后 indices 记录数：0
成分股明细统计：


,index_code,stock_count
0,000300,300
